# LangGraph：用四个节点编排静态工程查询

LangGraph 是 orchestration/runtime（编排与运行时），不是 Retriever。
本 notebook 使用同一份小型工程进度知识库和 InMemoryVectorStore，只展示最小必要图：

**START → retrieve → evidence_gate → answer / refuse → END**

图只有四个节点，没有 fallback、循环、LLM、Agent、长期 memory 或持久化 checkpointer。
重点是观察“何时回答、何时拒答”；检索质量仍由数据、embedding 和 route 决定。

## 1. 自包含的数据准备

为了让 notebook 独立运行，下面直接读取、切分并解析当前 TXT。
这里只实现 graph 示例需要的字段，不导入项目内检索/可靠性模块。

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Any, TypedDict

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph


repo_root = Path.cwd().resolve()
if repo_root.name == "ZZworkbench":
    repo_root = repo_root.parent

text_dir = repo_root / "knowledge" / "project_progress" / "texts" / "v4"
embed_path = (
    Path("/mnt/e/local_models/embedding")
    / "iic--nlp_gte_sentence-embedding_chinese-base"
)
assert text_dir.is_dir()
assert embed_path.is_dir()
print({"repo_ok": repo_root.name == "pipelines_rag", "model": embed_path.name})

{'repo_ok': True, 'model': 'iic--nlp_gte_sentence-embedding_chinese-base'}


In [2]:
title_pattern = re.compile(r"该进度计划的完整名称为(?P<title>.+?)[。\n]")
documents: list[Document] = []

for path in sorted(text_dir.glob("*.txt")):
    text = path.read_text(encoding="utf-8-sig").replace("\r\n", "\n").strip()
    relative_source = path.relative_to(repo_root).as_posix()
    document_id = "doc-" + hashlib.sha1(
        f"{relative_source}\n{text}".encode("utf-8")
    ).hexdigest()[:16]
    title_match = title_pattern.search(text)
    documents.append(
        Document(
            id=document_id,
            page_content=text,
            metadata={
                "source_name": path.name,
                "document_id": document_id,
                "title": (
                    title_match.group("title").strip()
                    if title_match
                    else path.stem
                ),
            },
        )
    )

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "。", "！", "？", "；", "，", " ", ""],
    keep_separator="end",
    chunk_size=872,
    chunk_overlap=160,
    add_start_index=True,
)
chunks: list[Document] = []
for document in documents:
    for index, chunk in enumerate(splitter.split_documents([document])):
        chunk_id = (
            f"{document.id}:"
            f"{int(chunk.metadata.get('start_index', 0)):06d}:{index:03d}"
        )
        chunks.append(
            Document(
                id=chunk_id,
                page_content=chunk.page_content,
                metadata={**chunk.metadata, "chunk_id": chunk_id},
            )
        )

chunks_by_document: dict[str, list[Document]] = defaultdict(list)
for chunk in chunks:
    chunks_by_document[chunk.metadata["document_id"]].append(chunk)

page_pattern = re.compile(r"以下内容来自PDF第(?P<page>\d+)页。")
task_pattern = re.compile(r"^标识号(?P<task_id>\d+)是")
quoted_pattern = re.compile(r"“(?P<value>[^”]+)”")
start_pattern = re.compile(r"计划开始(?P<value>\d{4}年\d{1,2}月\d{1,2}日)")
end_pattern = re.compile(r"计划完成(?P<value>\d{4}年\d{1,2}月\d{1,2}日)")
duration_pattern = re.compile(r"工期(?P<value>[^。]+)")

records: list[dict] = []
for document in documents:
    page: int | None = None
    document_chunks = chunks_by_document[document.metadata["document_id"]]
    for paragraph in re.split(r"\n\s*\n", document.page_content):
        paragraph = paragraph.strip()
        if page_match := page_pattern.fullmatch(paragraph):
            page = int(page_match.group("page"))
            continue
        task_match = task_pattern.match(paragraph)
        if task_match is None:
            continue
        quoted = [match.group("value") for match in quoted_pattern.finditer(paragraph)]
        if not quoted:
            continue
        task_id = task_match.group("task_id")
        task_name = quoted[-1]
        chunk = next(
            (candidate for candidate in document_chunks if paragraph in candidate.page_content),
            None,
        )
        if chunk is None:
            chunk = next(
                (
                    candidate
                    for candidate in document_chunks
                    if f"标识号{task_id}是" in candidate.page_content
                    and task_name in candidate.page_content
                ),
                None,
            )
        start_match = start_pattern.search(paragraph)
        end_match = end_pattern.search(paragraph)
        duration_match = duration_pattern.search(paragraph)
        header = paragraph.split("。", 1)[0]
        is_child = "子任务" in header
        is_parent = "同时是父任务" in header or (
            header.startswith(f"标识号{task_id}是父任务") and not is_child
        )
        records.append(
            {
                "source_name": document.metadata["source_name"],
                "project_title": document.metadata["title"],
                "task_id": task_id,
                "task_name": task_name,
                "chunk_id": str(chunk.id) if chunk else None,
                "page": page,
                "is_parent": is_parent,
                "start_date": start_match.group("value") if start_match else None,
                "end_date": end_match.group("value") if end_match else None,
                "duration": (
                    duration_match.group("value").strip()
                    if duration_match
                    else None
                ),
            }
        )

assert all(record["chunk_id"] for record in records)
print({"documents": len(documents), "chunks": len(chunks), "records": len(records)})

{'documents': 8, 'chunks': 63, 'records': 541}


In [3]:
embeddings = HuggingFaceEmbeddings(
    model=str(embed_path),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
    query_encode_kwargs={"normalize_embeddings": True},
    show_progress=False,
)
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(
    documents=chunks,
    ids=[str(chunk.id) for chunk in chunks],
)
print({"store": type(vector_store).__name__, "indexed_chunks": len(chunks)})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'store': 'InMemoryVectorStore', 'indexed_chunks': 63}


## 2. 最小 route 与查询规则

工程 route 使用当前示例实际需要的、可审计的别名；没有明确工程时不猜。
任务名优先从已解析 records 中做精确词面识别，失败后才剥离常见问句后缀。

In [4]:
source_aliases = {
    "110kV黄金输变电工程三级进度计划.txt": [
        "黄金", "黄金输变电工程", "110千伏黄金输变电工程",
    ],
    "110千伏节点计划-重点关注.txt": [
        "节点计划", "110千伏输变电工程节点计划",
    ],
    "三级进度计划-土建.txt": ["禾益", "禾益输变电工程"],
    "三虎输变电工程三级进度计划土建部分.txt": [
        "三虎土建", "三虎输变电工程土建部分",
    ],
    "三虎输变电工程三级进度计划电气部分.txt": [
        "三虎电气", "三虎输变电工程电气部分",
    ],
    "南溪三级进度.txt": [
        "南溪", "南溪输变电工程", "南溪旅游输变电工程",
    ],
    "珠海110kV江湾输变电工程总体进度计划横道图.txt": [
        "江湾总体计划", "江湾输变电工程总体计划",
    ],
    "珠海110千伏江湾输变电工程施工进度计划（202.txt": [
        "江湾施工进度计划", "江湾输变电工程施工进度计划",
    ],
}


def normalize_query(text: str) -> str:
    normalized = unicodedata.normalize("NFKC", text).casefold()
    normalized = re.sub(
        r"(?P<voltage>\d+)\s*k\s*v",
        r"\g<voltage>千伏",
        normalized,
    )
    normalized = normalized.replace("签订", "签定")
    return re.sub(r"[^0-9a-z\u3400-\u9fff]+", "", normalized)


def resolve_sources(query: str) -> list[str]:
    normalized = normalize_query(query)
    scores = {
        source: max(
            (
                len(normalized_alias)
                for alias in aliases
                if (normalized_alias := normalize_query(alias))
                and normalized_alias in normalized
            ),
            default=0,
        )
        for source, aliases in source_aliases.items()
    }
    best = max(scores.values(), default=0)
    return sorted(
        source for source, score in scores.items() if best > 0 and score == best
    )

## 3. Graph State

State 只保存跨节点需要的数据。hits 已经是普通 dict，不再建立
serialize_hit / deserialize_hit 两套适配。

In [5]:
class RAGState(TypedDict, total=False):
    question: str
    normalized_query: str
    project_sources: list[str]
    requested_fields: list[str]
    task_hint: str | None
    hits: list[dict[str, Any]]
    status: str
    record: dict[str, Any] | None
    answer: str
    citations: list[dict[str, Any]]

## 4. 四个节点

- retrieve：规范化、工程过滤、Dense with_score；
- evidence_gate：唯一任务记录、字段、引用和候选覆盖检查；
- answer：只格式化 exact 证据；
- refuse：根据 gate 状态给出澄清或拒答。

这里故意不做 fallback。没有证据时直接拒答，比让图反复改写查询更适合当前单跳场景。

In [6]:
def retrieve_node(state: RAGState) -> dict[str, Any]:
    normalized = normalize_query(state["question"])
    project_sources = resolve_sources(state["question"])
    allowed = set(project_sources) if project_sources else None
    source_filter = lambda doc: (
        allowed is None or doc.metadata["source_name"] in allowed
    )
    pairs = vector_store.similarity_search_with_score(
        normalized,
        k=8,
        filter=source_filter,
    )
    return {
        "normalized_query": normalized,
        "project_sources": project_sources,
        "hits": [
            {
                "chunk_id": str(doc.id),
                "source_name": doc.metadata["source_name"],
                "score": float(score),
                "text": doc.page_content,
            }
            for doc, score in pairs
        ],
    }


def evidence_gate_node(state: RAGState) -> dict[str, Any]:
    scoped = [
        record
        for record in records
        if not state["project_sources"]
        or record["source_name"] in state["project_sources"]
    ]
    exact_names = {
        record["task_name"]
        for record in scoped
        if normalize_query(record["task_name"]) in state["normalized_query"]
    }
    task_hint = (
        max(exact_names, key=lambda name: len(normalize_query(name)))
        if exact_names
        else None
    )
    if task_hint is None:
        segment = state["question"].split("的")[-1]
        for suffix in [
            r"计划什么时候完成.*$",
            r"什么时候完成.*$",
            r"计划起止时间是什么.*$",
            r"在什么时间.*$",
        ]:
            segment = re.sub(suffix, "", segment).strip("？?。 ，,")
        task_hint = re.sub(r"计划$", "", segment).strip() or None

    requested_fields = []
    if any(word in state["normalized_query"] for word in ["开始", "起止", "在什么时间"]):
        requested_fields.append("start_date")
    if any(word in state["normalized_query"] for word in ["完成", "结束", "起止", "在什么时间"]):
        requested_fields.append("end_date")
    if any(word in state["normalized_query"] for word in ["多久", "工期"]):
        requested_fields.append("duration")
    if not requested_fields:
        requested_fields = ["start_date", "end_date"]

    matches = [
        record
        for record in scoped
        if task_hint
        and normalize_query(record["task_name"]) == normalize_query(task_hint)
    ]
    if "父任务" in state["normalized_query"]:
        parent_matches = [record for record in matches if record["is_parent"]]
        if parent_matches:
            matches = parent_matches

    base = {
        "task_hint": task_hint,
        "requested_fields": requested_fields,
        "record": None,
        "citations": [],
    }
    if not task_hint:
        return {**base, "status": "insufficient"}
    if not matches:
        return {**base, "status": "not_found"}
    if len({record["source_name"] for record in matches}) > 1 or len(matches) > 1:
        return {**base, "status": "ambiguous"}

    record = matches[0]
    if any(not record.get(field) for field in requested_fields):
        return {**base, "status": "insufficient", "record": record}
    if any(not record.get(field) for field in ["source_name", "task_id", "chunk_id"]):
        return {**base, "status": "insufficient", "record": record}
    hit_ids = {hit["chunk_id"] for hit in state["hits"]}
    if record["chunk_id"] not in hit_ids:
        return {**base, "status": "insufficient", "record": record}

    citation = {
        "source_name": record["source_name"],
        "task_id": record["task_id"],
        "chunk_id": record["chunk_id"],
        "page": record["page"],
    }
    return {
        **base,
        "status": "exact",
        "record": record,
        "citations": [citation],
    }


def answer_node(state: RAGState) -> dict[str, Any]:
    record = state["record"]
    labels = {
        "start_date": "计划开始",
        "end_date": "计划完成",
        "duration": "工期",
    }
    facts = "，".join(
        f"{labels[field]}{record[field]}"
        for field in state["requested_fields"]
    )
    citation = state["citations"][0]
    citation_text = (
        f"来源：{citation['source_name']}，任务标识号{citation['task_id']}"
        + (
            f"，PDF第{citation['page']}页"
            if citation["page"] is not None
            else ""
        )
        + f"，chunk_id={citation['chunk_id']}"
    )
    return {
        "answer": (
            f"{record['project_title']}中，"
            f"“{record['task_name']}”{facts}。\n{citation_text}"
        )
    }


def refuse_node(state: RAGState) -> dict[str, Any]:
    task = state.get("task_hint") or "未识别任务"
    messages = {
        "ambiguous": f"任务“{task}”存在多个候选，请补充工程或父子层级。",
        "not_found": f"知识库中没有找到任务“{task}”的可验证记录。",
        "insufficient": f"任务“{task}”的字段、引用或召回证据不足，不能回答。",
    }
    return {"answer": messages.get(state["status"], "证据不足，不能回答。")}

## 5. 构图与条件边

evidence_gate 之后只做一次二选一：exact 进入 answer，其余状态进入 refuse。
这已经足够展示 Conditional Edge，不需要为每个拒答状态创建单独节点。

In [7]:
builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("evidence_gate", evidence_gate_node)
builder.add_node("answer", answer_node)
builder.add_node("refuse", refuse_node)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "evidence_gate")
builder.add_conditional_edges(
    "evidence_gate",
    lambda state: "answer" if state["status"] == "exact" else "refuse",
    {"answer": "answer", "refuse": "refuse"},
)
builder.add_edge("answer", END)
builder.add_edge("refuse", END)

graph = builder.compile()
print({"nodes": ["retrieve", "evidence_gate", "answer", "refuse"]})

{'nodes': ['retrieve', 'evidence_gate', 'answer', 'refuse']}


## 6. 三条真实路径

success、ambiguous、not_found 都来自当前 v4。stream 输出实际经过的 node，
可以看到成功与拒答在 evidence_gate 后分流。

In [8]:
graph_queries = {
    "success": "珠海110kV黄金输变电工程的主体结构封顶计划什么时候完成？",
    "ambiguous": "施工准备什么时候完成？",
    "not_found": "南溪输变电工程的锅炉点火计划什么时候完成？",
}

graph_results = {}
for name, question in graph_queries.items():
    updates = list(
        graph.stream({"question": question}, stream_mode="updates")
    )
    result = graph.invoke({"question": question})
    graph_results[name] = result
    print(
        json.dumps(
            {
                "case": name,
                "path": [next(iter(update)) for update in updates],
                "status": result["status"],
                "answer": result["answer"],
            },
            ensure_ascii=False,
            indent=2,
        )
    )

{
  "case": "success",
  "path": [
    "retrieve",
    "evidence_gate",
    "answer"
  ],
  "status": "exact",
  "answer": "珠海110千伏黄金输变电工程工程进度计划横道图中，“主体结构封顶”计划完成2024年10月24日。\n来源：110kV黄金输变电工程三级进度计划.txt，任务标识号10，PDF第1页，chunk_id=doc-b94ad48bc28360c5:000000:000"
}


{
  "case": "ambiguous",
  "path": [
    "retrieve",
    "evidence_gate",
    "refuse"
  ],
  "status": "ambiguous",
  "answer": "任务“施工准备”存在多个候选，请补充工程或父子层级。"
}
{
  "case": "not_found",
  "path": [
    "retrieve",
    "evidence_gate",
    "refuse"
  ],
  "status": "not_found",
  "answer": "知识库中没有找到任务“锅炉点火”的可验证记录。"
}


## 7. Graph 与直接函数应给出相同结果

LangGraph 不应悄悄改变业务判断。下面用同样四步直接调用节点，并检查状态、答案和引用
与 graph.invoke 一致。

In [9]:
def direct_query(question: str) -> RAGState:
    state: RAGState = {"question": question}
    state.update(retrieve_node(state))
    state.update(evidence_gate_node(state))
    if state["status"] == "exact":
        state.update(answer_node(state))
    else:
        state.update(refuse_node(state))
    return state


comparison = []
for name, question in graph_queries.items():
    direct = direct_query(question)
    graphed = graph_results[name]
    same = (
        direct["status"] == graphed["status"]
        and direct["answer"] == graphed["answer"]
        and direct["citations"] == graphed["citations"]
    )
    comparison.append({"case": name, "same": same})

print(comparison)
assert all(row["same"] for row in comparison)

[{'case': 'success', 'same': True}, {'case': 'ambiguous', 'same': True}, {'case': 'not_found', 'same': True}]


## 8. 为什么核心流程不加 checkpoint

InMemorySaver 可以按 thread_id 保存 graph state，但当前查询是单次、无人工中断的静态查找，
进程结束后也不需要恢复。把 checkpoint 放进主流程只会增加概念负担。

当未来真的出现人工审核、长任务暂停恢复或多步骤审批，再单独引入 checkpointer。
checkpoint 解决执行状态恢复，不是长期知识记忆，也不会提高召回准确率。

## 9. 结论

- 四节点已经足够表达检索、证据判断、回答和拒答；
- State 让步骤输入输出可观察，Conditional Edge 让拒答路径可测试；
- 对当前单跳事实查询，直接函数调用更短，仍应作为默认实现；
- 只有出现真实分支、恢复、人工审核或多步骤状态需求时，LangGraph 才带来净收益；
- 小型知识库继续使用 InMemoryVectorStore，无需持久化向量数据库。